# kluster.ai

[kluster.ai](https://kluster.ai) is a inference service that provides access to a variety of high-performance LLMs including Meta's Llama 3.1 and Llama 3.3 models, DeepSeek R1 and V3, Gemma 3 27B and more. You can find all of [kluster.ai's available models](https://docs.kluster.ai/get-started/models/) in the documentation.

kluster.ai provides an API through which developers can perform [real-time](https://docs.kluster.ai/get-started/start-building/real-time/) and [batch](https://docs.kluster.ai/get-started/start-building/batch/) inference, [fine-tune]() their models, and more. For details of all kluster.ai API features, head to the [API reference](https://docs.kluster.ai/api-reference/reference/).

This notebook goes over how to use LangChain with kluster.ai for language models via the `chat.invoke` endpoint.

## Prerequisites

Before getting started, ensure you have the following:

- **A kluster.ai account** - sign up on the <a href="https://platform.kluster.ai/signup" target="_blank">kluster.ai platform</a> if you don't have one
- **A kluster.ai API key** - after signing in, go to the <a href="https://platform.kluster.ai/apikeys" target="_blank">**API Keys**</a> section and create a new key. For detailed instructions, check out the <a href="/get-started/get-api-key/" target="_blank">Get an API key</a> guide
- **LangChain community and core installed** - you can install them by running:


## Setup

In this notebook, we'll use Python's `getpass` module to safely input the key. After execution, please provide your unique kluster.ai API key (ensure no spaces).

In [1]:
# Get a new token from https://platform.kluster.ai if not already set
import os
from getpass import getpass

# Only prompt for API token if not already set in environment
if "KLUSTERAI_API_KEY" not in os.environ:
    print("Please enter your kluster.ai API token:")
    KLUSTERAI_API_TOKEN = getpass()
    os.environ["KLUSTERAI_API_KEY"] = KLUSTERAI_API_TOKEN
else:
    print("Using existing KLUSTERAI_API_KEY from environment")

Please enter your kluster.ai API token:


 ········


With the API key already set, import `KlusterAI` from LangChain community chat models, and initialize it by passing the model. This example uses `klusterai/Meta-Llama-3.1-8B-Instruct-Turbo`, but feel free to try any other of the [supported models](https://docs.kluster.ai/get-started/models/).

In [2]:
from langchain_community.llms import KlusterAI

# Initialize llm with model
llm = KlusterAI(model_id="klusterai/Meta-Llama-3.1-8B-Instruct-Turbo")

## Invocation

Once you've instantiated the `llm` client, you can easily create a real-time inference with the `invoke` method.

In [3]:
# Invoke with message directly
llm.invoke("Who let the dogs out?")

'The phrase "Who let the dogs out?" comes from a song of the same name by the American rap-rock band Baha Men. The song was released in 2000 and became a massive hit worldwide, particularly at sporting events and parties, due to its catchy chorus and energetic vibe.\n\nThe song is reported to be an adaptation of a Shaggy 2 Dope song from the group Insane Clown Posse, with some parts rewritten and new parts added.\n\nAs for the answer to your question, the song itself doesn\'t explain who let the dogs out, so it\'s left to the listener\'s imagination.'

### Using different models

kluster.ai offers many models that you can use with LangChain. You can find all of [kluster.ai's available models](https://docs.kluster.ai/get-started/models/) in the documentation.

To use a different model, replace the model API name in the `model` parameter.

In [4]:
# Using the larger 405B parameter model
large_model_llm = KlusterAI(
    model_id="klusterai/Meta-Llama-3.1-405B-Instruct-Turbo"
)

# Different model invoke
large_model_llm.invoke("Who let the dogs out?")

"The phrase 'Who let the dogs out' is the title of a song written and performed by the Bahamian group Baha Men. The song was released in 2000 and became a worldwide hit, topping the charts in many countries and being featured in various sporting events, movies, and commercials."

### Controlling model parameters

In addition to providing a `model`, you can control various parameters such as `temperature` to adjust the model's creativity level. For more parameters, check kluster.ai API [chat completion reference](https://docs.kluster.ai/api-reference/reference/#create-chat-completion).

In [5]:
# Initialize llm with model and adjust parameters
precise_llm = KlusterAI(model_id="klusterai/Meta-Llama-3.1-8B-Instruct-Turbo")
precise_llm.model_kwargs = {
    "temperature": 0.1,
    "max_tokens": 250,
    "top_p": 0.9,
}

# Invoke with message directly
llm.invoke("Who let the dogs out?")

'A classic song reference. "Who Let the Dogs Out?" is a song written and originally recorded by Bahamian musician Anslem Douglas in 1998. However, the version that became well-known worldwide was a dancehall-ragga remix of the song, also known as the "Baha Men" version, which was released in 2000.'

## Streaming functionality

`KlusterAI` also supports streaming functionalities via the `stream` endpoint.

In [6]:
# Example of streaming functionality
for chunk in llm.stream("Who let the dogs out?"):
    print(chunk, end="")

A classic song title.  The song "Who Let the Dogs Out" is by a band called Baha Men.

## Using `PromptTemplates` with `KlusterAI`

We will create a prompt template for Question and Answer.


You can combine `KlusterAI` with LangChain's `PromptTemplate` for more structured interactions. The following example creates a template for a question/answer based query, in which the model is asked to provide a broken down answer.

In [7]:
from langchain_core.prompts import PromptTemplate

# Define prompt message with placeholders
template = """Question: {question}
Answer: Let's think step by step."""

# Create prompt template
prompt = PromptTemplate.from_template(template)

# Format the prompt with input value
formatted_prompt = prompt.format(question="Who let the dogs out?")

# Now you can pass `formatted_prompt` to a chat model manually
llm.invoke(formatted_prompt)

'To solve this question, "Who let the dogs out?" which seems to be an indirect reference to the song "Who Let the Dogs Out?" by Baha Men, we can analyze it as an attempt at a riddle or a humorous anecdote:\n\n1. The question itself, "Who let the dogs out?" implies a story that requires a resolution.\n2. Knowing the song origin might be necessary - the song determines the phrase to be used when someone needs an audience/ a crowd of people having some time let off during the party to finalise allowing animals escaped regained control of that party/reason.\n\n\n\nIdeally linking two good frenemies or practically linked individuals could avoid dogs needlessly roam and encincible people we trust entered somebody having loved (between owners leaving themselves leftover stinking) something to looks themselves diffident touching lax incline in comfort and excessively excellent proper conjunction reference written developed at its light released occurred their faithful fault lifted alteration r

### Create a runnable sequence

You can also pipe the template directly into the chat model with the pipe operators to create a `RunnableSequence`.

In [8]:
# Pipe the template directly into the llm model
chain = prompt | llm

# Run the chain
chain.invoke({"question": "Can penguins reach the North pole?"})


"Let's break down the question and consider the feasibility of penguins reaching the North Pole.\n\n1. **Habitat and Migration**: Penguins are found in the Southern Hemisphere, primarily in Antarctica and the surrounding islands, as well as in the southern parts of South America, Africa, and Australia. They are not native to the Northern Hemisphere.\n\n2. **Climate**: The North Pole is a frozen sea ice environment, with extremely cold temperatures, strong winds, and long, dark winters. Penguins are adapted to living in cold climates, but the conditions at the North Pole are far more extreme than what they experience in their native habitats.\n\n3. **Food Availability**: Penguins are primarily fed by fish, krill, and squid. The waters surrounding the North Pole are limited in terms of nutrient supply, particularly during the winter months. Penguins need access to a reliable food source to survive.\n\n4. **Flying ability**: Most penguin species can't fly, and even for those that can, lon